# (Figure) Example rastergrams.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from simulated_data import simulated_data

K = 50
T = 100

# 3.5 inch column-width figure
fig, axes = plt.subplots(
    2, 1,
    figsize=(3.5, 2.6),
    constrained_layout=True
)

# ---------- Top raster ----------
raster = np.zeros((K, T))
for k in range(K):
    raster[k, :] = simulated_data(T)

axes[0].imshow(
    raster,
    cmap="binary",
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest",
    origin="upper"
)

# scale bar (10 time bins) near lower left
axes[0].plot([5, 15], [K - 3, K - 3], color='k', linewidth=2.2, solid_capstyle='butt')
axes[0].axis("off")

# ---------- Bottom raster ----------
raster = np.zeros((K, T))
for k in range(K):
    raster[k, :] = simulated_data(T, 10, 50)

axes[1].imshow(
    raster,
    cmap="binary",
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest",
    origin="upper"
)

axes[1].plot([5, 15], [K - 3, K - 3], color='k', linewidth=2.2, solid_capstyle='butt')
axes[1].axis("off")

plt.savefig(
    "./ps/Fig5_A_Simulated_Rasters.pdf",
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.01
)
plt.show()

# Load global settings and functions.

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from Utilities import set_seed, append_row, read_best_params, compute_confusion_matrix
from train import TrainConfig, Trainer
from simulated_data import simulated_data

# ---------------- Global Settings ----------------
N_TRAIN    = 1000
N_VAL      = 200
N_TEST     = 200
BATCH_SIZE = 128
EPOCHS     = 30
EVAL_SEEDS = list(range(1, 1001))

def _make_split(n_samples: int):
    n_case0 = n_samples // 2
    n_case1 = n_samples - n_case0

    # Case 0: constant-rate Poisson spike train
    x0 = np.stack([simulated_data(100) for _ in range(n_case0)], axis=0)
    y0 = np.zeros(n_case0, dtype=np.int64)

    # Case 1: sinusoidally modulated Poisson spike train
    x1 = np.stack([simulated_data(100, 10, 50) for _ in range(n_case1)], axis=0)
    y1 = np.ones(n_case1, dtype=np.int64)

    x = np.concatenate([x0, x1], axis=0).astype(np.float32)
    y = np.concatenate([y0, y1], axis=0)

    perm = np.random.permutation(n_samples)
    return x[perm], y[perm]


def _to_loader(x, y, batch_size, shuffle, generator=None):
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )

def make_simulated_loaders(
    n_train: int,
    n_val: int,
    n_test: int,
    batch_size: int,
    seed: int,
):
    set_seed(seed)

    x_train, y_train = _make_split(n_train)
    x_val, y_val = _make_split(n_val)
    x_test, y_test = _make_split(n_test)

    train_gen = torch.Generator().manual_seed(seed)
    train_loader = _to_loader(x_train, y_train, batch_size, shuffle=True, generator=train_gen)
    val_loader = _to_loader(x_val, y_val, batch_size, shuffle=False)
    test_loader = _to_loader(x_test, y_test, batch_size, shuffle=False)

    split_info = {
        "n_total": int(n_train + n_val + n_test),
        "n_train": int(n_train),
        "n_val": int(n_val),
        "n_test": int(n_test),
        "seq_len": int(x_train.shape[1]),
    }
    return train_loader, val_loader, test_loader, split_info

# Run RRNs

In [ ]:
# ---------------- Local Settings ---------------- 
from RRN_functions import RRNModel, rrn_prepare_batch
FS     = 1000

BEST_CSV = "./out/best_runs/best_hparams_from_grid.csv"
OUT_DIR  = "./out/best_runs_simulated"
os.makedirs(OUT_DIR, exist_ok=True)

WRES_DIR = os.path.join(OUT_DIR, "W_res")
os.makedirs(WRES_DIR, exist_ok=True)

CM_DIR = os.path.join(OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "rrn_best_runs_simulated.csv")

ARCH_CONFIGS = [
    ("RRN_golden",             "golden",  False),
    ("RRN_log",                "log",     False),
    ("RRN_two",                "two",     False),
    ("RRN_linear",             "linear",  False),
]

if __name__ == '__main__':
    if os.path.exists(BEST_CSV):
        best = read_best_params(BEST_CSV)
    else:
        best = {}

    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    cols = [
        'arch_name', 'rhythm', 'no_recurrence',
        'seed',
        'n_total', 'n_train', 'n_val', 'n_test', 'seq_len',
        'lr', 'weight_decay', 'label_smoothing',
        'best_epoch', 'best_val_acc',
        'test_loss', 'test_acc',
        'w_res_path',
    ]

    for arch_name, rhythm, no_recurrence in ARCH_CONFIGS:
        hp = best.get(arch_name, {'lr': 1e-3, 'weight_decay': 1e-2, 'label_smoothing': 0.0})
        print(f"\n===== {arch_name} (simulated; rhythm={rhythm}, no_recurrence={no_recurrence}) =====")
        print('Using hparams:', hp)

        for seed in EVAL_SEEDS:
            set_seed(seed)

            train_loader, val_loader, test_loader, split_info = make_simulated_loaders(
                n_train=N_TRAIN,
                n_val=N_VAL,
                n_test=N_TEST,
                batch_size=BATCH_SIZE,
                seed=seed,
            )

            model = RRNModel(
                Fs=FS,
                n_classes=2,
                rhythm_configuration=rhythm,
                no_recurrence=no_recurrence,
                verbose=False,
            )

            cfg = TrainConfig(
                num_classes=2,
                epochs=EPOCHS,
                lr=hp['lr'],
                weight_decay=hp['weight_decay'],
                label_smoothing=hp['label_smoothing'],
                seed=seed,
                history_csv=False,
            )

            trainer = Trainer(
                cfg,
                prepare_batch=rrn_prepare_batch,
                class_names=['simulated_const', 'simulated_modulated'],
            )

            results = trainer.fit(
                model,
                train_loader,
                val_loader,
                test_loader,
                run_name=f"{arch_name}_simulated_seed{seed}",
            )

            w_res_path = ''
            if hasattr(model, 'W_res') and getattr(model.W_res, 'requires_grad', False):
                w_res_path = os.path.join(WRES_DIR, f"W_res_{arch_name}_seed{seed}.npy")
                np.save(w_res_path, model.W_res.detach().cpu().numpy())

            cm = compute_confusion_matrix(model, test_loader, rrn_prepare_batch, n_classes=2)
            cm_path = os.path.join(CM_DIR, f"confmat_{arch_name}_seed{seed}.npy")
            np.save(cm_path, cm)

            row = dict(
                arch_name=arch_name,
                rhythm=rhythm,
                no_recurrence=no_recurrence,
                seed=seed,
                n_total=split_info['n_total'],
                n_train=split_info['n_train'],
                n_val=split_info['n_val'],
                n_test=split_info['n_test'],
                seq_len=split_info['seq_len'],
                lr=hp['lr'],
                weight_decay=hp['weight_decay'],
                label_smoothing=hp['label_smoothing'],
                best_epoch=results['best_epoch'],
                best_val_acc=results['best_val_acc'],
                test_loss=results['test']['loss'],
                test_acc=results['test']['acc'],
                w_res_path=w_res_path,
            )
            append_row(OUT_CSV, cols, row)
            print(arch_name, 'seed', seed, '| test_acc', row['test_acc'], '| w_res', ('YES' if w_res_path else 'NO'))

    print('\nSaved results:', OUT_CSV)
    print('Saved W_res dir:', WRES_DIR)
    print('Saved confusion matrices dir:', CM_DIR)



# Run LSTM

In [ ]:
# ---------------- Local Settings ----------------
from LSTM_functions import LSTMModel, lstm_prepare_batch
HIDDEN_SIZES = [6]

BEST_CSV = './out/best_runs/best_hparams_from_grid.csv'
OUT_DIR = './out/best_runs_simulated_lstm'
os.makedirs(OUT_DIR, exist_ok=True)

CM_DIR = os.path.join(OUT_DIR, 'confusion_matrices')
os.makedirs(CM_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, 'lstm_best_runs_simulated.csv')

if __name__ == '__main__':
    if os.path.exists(BEST_CSV):
        best = read_best_params(BEST_CSV)
    else:
        best = {}

    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    cols = [
        'arch_name', 'hidden_size', 'seed',
        'n_total', 'n_train', 'n_val', 'n_test', 'seq_len',
        'lr', 'weight_decay', 'label_smoothing',
        'best_epoch', 'best_val_acc', 'test_loss', 'test_acc',
    ]

    for H in HIDDEN_SIZES:
        arch_name = f'LSTM_h{H}'
        hp = best.get(arch_name) # Get lr, weight_decay, label_smoothing
        print(f'\n===== {arch_name} (simulated, n_classes=2) =====')
        print('Using hparams:', hp)

        for seed in EVAL_SEEDS:
            set_seed(seed)

            train_loader, val_loader, test_loader, split_info = make_simulated_loaders(
                n_train=N_TRAIN,
                n_val=N_VAL,
                n_test=N_TEST,
                batch_size=BATCH_SIZE,
                seed=seed,
            )

            model = LSTMModel(n_classes=2, hidden_size=H)

            cfg = TrainConfig(
                num_classes=2,
                epochs=EPOCHS,
                lr=hp['lr'],
                weight_decay=hp['weight_decay'],
                label_smoothing=hp['label_smoothing'],
                seed=seed,
                history_csv=False,
            )

            trainer = Trainer(
                cfg,
                prepare_batch=lstm_prepare_batch,
                class_names=['simulated_const', 'simulated_modulated'],
            )

            results = trainer.fit(
                model,
                train_loader,
                val_loader,
                test_loader,
                run_name=f'{arch_name}_simulated_seed{seed}',
            )

            cm = compute_confusion_matrix(model, test_loader, lstm_prepare_batch, n_classes=2)
            cm_path = os.path.join(CM_DIR, f'confmat_{arch_name}_seed{seed}.npy')
            np.save(cm_path, cm)

            row = dict(
                arch_name=arch_name,
                hidden_size=H,
                seed=seed,
                n_total=split_info['n_total'],
                n_train=split_info['n_train'],
                n_val=split_info['n_val'],
                n_test=split_info['n_test'],
                seq_len=split_info['seq_len'],
                lr=hp['lr'],
                weight_decay=hp['weight_decay'],
                label_smoothing=hp['label_smoothing'],
                best_epoch=results['best_epoch'],
                best_val_acc=results['best_val_acc'],
                test_loss=results['test']['loss'],
                test_acc=results['test']['acc'],
            )
            append_row(OUT_CSV, cols, row)
            print(arch_name, 'seed', seed, '| test_acc', row['test_acc'])

    print('\nSaved results:', OUT_CSV)
    print('Saved confusion matrices dir:', CM_DIR)
    

# Run Standard RNN

In [ ]:
# ---------------- Local Settings ----------------
from RNN_functions import StandardRNN, rnn_prepare_batch
HIDDEN_SIZES = [11]

BEST_CSV = './out/best_runs/best_hparams_from_grid.csv'
OUT_DIR = './out/best_runs_simulated_rnn'
os.makedirs(OUT_DIR, exist_ok=True)

CM_DIR = os.path.join(OUT_DIR, 'confusion_matrices')
os.makedirs(CM_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, 'rnn_best_runs_simulated.csv')


if __name__ == '__main__':
    if os.path.exists(BEST_CSV):
        best = read_best_params(BEST_CSV)
    else:
        best = {}

    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    cols = [
        'arch_name', 'hidden_size', 'seed',
        'n_total', 'n_train', 'n_val', 'n_test', 'seq_len',
        'lr', 'weight_decay', 'label_smoothing',
        'best_epoch', 'best_val_acc', 'test_loss', 'test_acc',
    ]

    for H in HIDDEN_SIZES:
        arch_name = f'RNN_h{H}'
        hp = best.get(arch_name, {'lr': 1e-3, 'weight_decay': 1e-2, 'label_smoothing': 0.0})
        print(f'\n===== {arch_name} (simulated, n_classes=2) =====')
        print('Using hparams:', hp)

        for seed in EVAL_SEEDS:
            set_seed(seed)

            train_loader, val_loader, test_loader, split_info = make_simulated_loaders(
                n_train=N_TRAIN,
                n_val=N_VAL,
                n_test=N_TEST,
                batch_size=BATCH_SIZE,
                seed=seed,
            )

            model = StandardRNN(n_classes=2, hidden_size=H)

            cfg = TrainConfig(
                num_classes=2,
                epochs=EPOCHS,
                lr=hp['lr'],
                weight_decay=hp['weight_decay'],
                label_smoothing=hp['label_smoothing'],
                seed=seed,
                history_csv=False,
            )

            trainer = Trainer(
                cfg,
                prepare_batch=rnn_prepare_batch,
                class_names=['simulated_const', 'simulated_modulated'],
            )

            results = trainer.fit(
                model,
                train_loader,
                val_loader,
                test_loader,
                run_name=f'{arch_name}_simulated_seed{seed}',
            )

            cm = compute_confusion_matrix(model, test_loader, rnn_prepare_batch, n_classes=2)
            cm_path = os.path.join(CM_DIR, f'confmat_{arch_name}_seed{seed}.npy')
            np.save(cm_path, cm)

            row = dict(
                arch_name=arch_name,
                hidden_size=H,
                seed=seed,
                n_total=split_info['n_total'],
                n_train=split_info['n_train'],
                n_val=split_info['n_val'],
                n_test=split_info['n_test'],
                seq_len=split_info['seq_len'],
                lr=hp['lr'],
                weight_decay=hp['weight_decay'],
                label_smoothing=hp['label_smoothing'],
                best_epoch=results['best_epoch'],
                best_val_acc=results['best_val_acc'],
                test_loss=results['test']['loss'],
                test_acc=results['test']['acc'],
            )
            append_row(OUT_CSV, cols, row)
            print(arch_name, 'seed', seed, '| test_acc', row['test_acc'])

    print('\nSaved results:', OUT_CSV)
    print('Saved confusion matrices dir:', CM_DIR)



# Analysis and Plots

## Barplots

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from Utilities import row_normalize_confmat

sources = [
    {
        'source_name': 'RRN',
        'csv_path': './out/best_runs_simulated/rrn_best_runs_simulated.csv',
        'cm_dir': './out/best_runs_simulated/confusion_matrices',
    },
    {
        'source_name': 'LSTM',
        'csv_path': './out/best_runs_simulated_lstm/lstm_best_runs_simulated.csv',
        'cm_dir': './out/best_runs_simulated_lstm/confusion_matrices',
    },
    {
        'source_name': 'Standard RNN',
        'csv_path': './out/best_runs_simulated_rnn/rnn_best_runs_simulated.csv',
        'cm_dir': './out/best_runs_simulated_rnn/confusion_matrices',
    },
]


def _safe_group_key(source_name, arch_name):
    if arch_name is None:
        return source_name
    return str(arch_name)


def _collect_groups_from_csv(source_name, csv_path, cm_dir):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f'Missing CSV: {csv_path}')

    df = pd.read_csv(csv_path)
    if 'test_acc' not in df.columns:
        raise KeyError(f"Column 'test_acc' not found in {csv_path}")

    has_arch = 'arch_name' in df.columns
    has_seed = 'seed' in df.columns

    groups = []
    if has_arch:
        arch_values = [a for a in df['arch_name'].dropna().unique().tolist()]
        for arch in arch_values:
            if str(arch) == 'RRN_golden_norecurrent':
                continue

            sub = df[df['arch_name'] == arch]
            vals = sub['test_acc'].dropna().to_numpy(dtype=float)
            if vals.size == 0:
                continue
            seed_set = set(sub['seed'].dropna().astype(int).tolist()) if has_seed else None
            groups.append({
                'label': _safe_group_key(source_name, arch),
                'source_name': source_name,
                'arch_name': str(arch),
                'acc': vals,
                'seeds': seed_set,
                'cm_dir': cm_dir,
            })
    else:
        vals = df['test_acc'].dropna().to_numpy(dtype=float)
        if vals.size > 0:
            seed_set = set(df['seed'].dropna().astype(int).tolist()) if has_seed else None
            groups.append({
                'label': source_name,
                'source_name': source_name,
                'arch_name': None,
                'acc': vals,
                'seeds': seed_set,
                'cm_dir': cm_dir,
            })

    return groups


def _find_group(groups_by_label, key):
    if key in groups_by_label:
        return groups_by_label[key]

    if key == 'LSTM':
        cands = [g for k, g in groups_by_label.items() if k.startswith('LSTM')]
        return cands[0] if cands else None

    if key == 'Standard RNN':
        cands = [g for k, g in groups_by_label.items() if k.startswith('RNN_')]
        return cands[0] if cands else None

    low = key.lower()
    cands = [g for k, g in groups_by_label.items() if low in k.lower()]
    return cands[0] if cands else None


all_groups = []
for s in sources:
    all_groups.extend(_collect_groups_from_csv(s['source_name'], s['csv_path'], s['cm_dir']))

if not all_groups:
    raise RuntimeError('No model groups found in the provided CSV files.')

for g in all_groups:
    g['mean_test_acc'] = float(np.mean(g['acc']))
    g['sem_test_acc'] = float(np.std(g['acc'], ddof=1) / np.sqrt(len(g['acc']))) if len(g['acc']) > 1 else np.nan

# Sort once and use this order for BOTH bar plot and confusion matrices.
all_groups = sorted(all_groups, key=lambda g: g['mean_test_acc'], reverse=True)

# ---------------- Accuracy plot + stats ----------------
labels = [g['label'] for g in all_groups]
means = np.array([g['mean_test_acc'] for g in all_groups], dtype=float)
sems = np.array([g['sem_test_acc'] for g in all_groups], dtype=float)

x = np.arange(len(labels))
colors = ['gold' if label == 'RRN_golden' else 'gray' for label in labels]

plt.figure(figsize=(4, 3.25))
plt.bar(x, means, color=colors, alpha=0.9)
valid_sem = np.isfinite(sems)
if np.any(valid_sem):
    plt.errorbar(
        x[valid_sem], means[valid_sem], yerr=2.0 * sems[valid_sem],
        fmt='none', ecolor='black', elinewidth=2, capsize=0, zorder=5,
    )

plt.xticks(x, labels, rotation=30, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.ylabel('Test Accuracy', fontsize=11)
plt.grid(axis='y', alpha=0.35, linestyle='--')
plt.ylim([0.5, 0.65])
plt.tight_layout()

print('Mean +/- SEM test accuracy (sorted):')
for g in all_groups:
    sem_txt = f"{g['sem_test_acc']:.4f}" if np.isfinite(g['sem_test_acc']) else 'nan'
    print(f"  {g['label']}: {g['mean_test_acc']:.4f} +/- {sem_txt} (n={len(g['acc'])})")

# OLS with RRN_golden as baseline (same style as RRN_Results.ipynb)
import statsmodels.formula.api as smf

GROUPS = ['RRN_golden', 'LSTM_h6', 'RRN_linear', 'RRN_two', 'RRN_log', 'RNN_h11']

# Build a long-form dataframe from all_groups for OLS.
df_ols = pd.DataFrame(
    [
        {'arch_name': g['label'], 'test_acc': float(v)}
        for g in all_groups
        for v in g['acc']
    ]
)

present_groups = [g for g in GROUPS if g in df_ols['arch_name'].unique()]

df_lm = df_ols[df_ols['arch_name'].isin(present_groups)].copy()
df_lm['group'] = pd.Categorical(df_lm['arch_name'], categories=present_groups)

if 'RRN_golden' not in present_groups:
    print('\nOLS skipped: RRN_golden not found in available groups.')
else:
    model = smf.ols(
        'test_acc ~ C(group, Treatment(reference="RRN_golden"))',
        data=df_lm
    ).fit()

    #print('\nOLS model summary:')
    #print(model.summary())

    out = pd.DataFrame({
        'predictor': model.params.index,
        'coef': model.params.values,
        'sem': model.bse.values,
        'p_value': model.pvalues.values,
    })

    out['__is_intercept__'] = out['predictor'].str.contains('Intercept')
    out = (
        out.sort_values(['__is_intercept__', 'predictor'], ascending=[False, True])
           .drop(columns='__is_intercept__')
    )

    print('\nOLS coefficients:')
    print(out.to_string(index=False, float_format=lambda x: f'{x:.3g}'))

plt.savefig("./ps/Fig5_B_Simulated_Boxplots.pdf", format="pdf", bbox_inches="tight", pad_inches=0)


## Confusion matrices

In [ ]:
# ---------------- Average confusion matrices per group ----------------
class_names = ['Constant', 'Modulated']
seed_re = re.compile(r'_seed(\d+)\.npy$')

results = []

for g in all_groups:
    cm_dir = g['cm_dir']
    arch = g['arch_name']

    if arch is None:
        cm_files = sorted(glob.glob(os.path.join(cm_dir, 'confmat_*.npy')))
    else:
        cm_files = sorted(glob.glob(os.path.join(cm_dir, f'confmat_{arch}_seed*.npy')))

    if g['seeds']:
        filtered = []
        for f in cm_files:
            m = seed_re.search(os.path.basename(f))
            if m is None:
                continue
            if int(m.group(1)) in g['seeds']:
                filtered.append(f)
        cm_files = filtered

    if not cm_files:
        print(f"\n{g['label']}: no confusion matrices found in {cm_dir} for arch={arch}")
        continue

    cms = []
    for f in cm_files:
        cm = np.load(f)
        if cm.shape != (2, 2):
            raise ValueError(f'Expected 2x2 confusion matrix in {f}, got {cm.shape}')
        cms.append(row_normalize_confmat(cm.astype(float)))

    cms = np.stack(cms, axis=0)
    cm_mean = cms.mean(axis=0)
    cm_sem = (cms.std(axis=0, ddof=1) / np.sqrt(cms.shape[0])) if cms.shape[0] > 1 else np.zeros_like(cm_mean)

    results.append({
        'label': g['label'],
        'n': cms.shape[0],
        'cm_mean': cm_mean,
        'cm_sem': cm_sem,
    })

for r in results:
    print(f"\n{r['label']} average confusion matrix (row-normalized, n={r['n']}):")
    for i in range(2):
        row_text = ' | '.join([f"{r['cm_mean'][i, j]:.2f} +/- {r['cm_sem'][i, j]:.3f}" for j in range(2)])
        print(f"  row {i} ({class_names[i]} true): {row_text}")

n_models = len(results)
if n_models == 0:
    raise RuntimeError('No confusion matrices found for any model group.')

ncols = min(6, n_models)
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(7.5, 1.75 * nrows),
    constrained_layout=True
)

axes = np.atleast_1d(axes).ravel()
tick_fs = 6
cmap = plt.cm.Blues
norm = plt.Normalize(vmin=0.0, vmax=1.0)

for k, r in enumerate(results):
    ax = axes[k]
    cm_mean = r['cm_mean']
    cm_sem = r['cm_sem']

    # Draw each cell as a vector rectangle instead of using imshow
    for i in range(2):
        for j in range(2):
            val = float(cm_mean[i, j])
            facecolor = cmap(norm(val))

            rect = plt.Rectangle(
                (j - 0.5, i - 0.5), 1, 1,
                facecolor=facecolor,
                edgecolor='black',
                linewidth=0.8
            )
            ax.add_patch(rect)

            err = float(cm_sem[i, j])
            text_color = 'white' if val >= 0.5 else 'black'
            ax.text(
                j, i,
                f'{val:.2f}\n±{err:.3f}',
                ha='center', va='center',
                color=text_color, fontsize=7
            )

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(1.5, -0.5)
    ax.set_aspect('equal')

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(class_names, fontsize=tick_fs)
    ax.set_yticklabels(class_names, fontsize=tick_fs)

    if k % ncols == 0:
        ax.set_ylabel('True', fontsize=9)
        ax.set_yticklabels(class_names, fontsize=tick_fs, rotation=90, va='center')
    else:
        ax.set_ylabel('')
        ax.set_yticklabels([])

    ax.set_title(r['label'], fontsize=9, pad=6)
    ax.tick_params(axis='both', which='major', labelsize=tick_fs, width=1.2, length=5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.4)
        spine.set_color('black')

for k in range(n_models, len(axes)):
    axes[k].axis('off')

fig.supxlabel('Predicted', fontsize=10)

plt.savefig(
    "./ps/Fig5_C_Confusion_Matrices.pdf",
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.02
)
plt.show()

## Spectra

In [ ]:
import os
import re
import glob
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from RRN_functions import RRNModel, rrn_extract_features
from simulated_data import simulated_data

# Load trained RRN_golden checkpoints from simulated-data runs
CKPT_DIR = './out/checkpoints'
ckpt_paths = sorted(glob.glob(os.path.join(CKPT_DIR, 'RRN_golden_simulated_seed*_best.pt')))
if not ckpt_paths:
    raise FileNotFoundError(
        'No checkpoints found matching ./out/checkpoints/RRN_golden_simulated_seed*_best.pt'
    )

print('Found', len(ckpt_paths), 'RRN_golden checkpoints')

model = RRNModel(Fs=1000, n_classes=2, rhythm_configuration='golden', verbose=False)
model.eval()

scenarios = [
    ('Const-rate Poisson', lambda: simulated_data(100)),
    ('Modulated Poisson (10-50 Hz)', lambda: simulated_data(100, 10, 50)),
]

Kreps = 1000
rng = np.random.default_rng(123)

mn = np.zeros((model.K, len(scenarios)), dtype=float)
sem = np.zeros((model.K, len(scenarios)), dtype=float)

with torch.no_grad():
    for j, (name, sampler) in enumerate(scenarios):
        print('Scenario:', name)
        F = np.zeros((model.K, Kreps), dtype=float)

        for k in range(Kreps):
            ckpt_path = ckpt_paths[int(rng.integers(len(ckpt_paths)))]
            state = torch.load(ckpt_path, map_location='cpu')
            model.load_state_dict(state['model_state'])

            x = sampler().astype(np.float32)
            f = rrn_extract_features(model, x)
            F[:, k] = f.cpu().numpy()

        mn[:, j] = np.mean(F, axis=1)
        sd = np.std(F, axis=1, ddof=1)
        sem[:, j] = sd / np.sqrt(Kreps)

fr = np.asarray(np.round(model.frange, 1), dtype=float)

# 2.5 inch wide figure
fig, ax = plt.subplots(figsize=(2.5, 2.2), dpi=120)

for j, (name, _) in enumerate(scenarios):
    m = mn[:, j]
    s = sem[:, j]
    ax.semilogx(model.frange, m, lw=1.4, label=name)
    ax.fill_between(model.frange, m - 1.96 * s, m + 1.96 * s, alpha=0.18)

# Show every frequency as an x-tick label
ax.set_xticks(fr)
ax.set_xticklabels([f'{x:.1f}' for x in fr], rotation=90)
ax.xaxis.set_minor_locator(mticker.NullLocator())

ax.set_yscale('log')
ax.yaxis.set_minor_locator(mticker.NullLocator())

ax.set_xlabel('Frequency (Hz)', fontsize=8)
ax.set_ylabel('Accumulated Amplitude Squared (a.u.)', fontsize=8)
ax.grid(True, which='both', alpha=0.25)
# ax.legend(fontsize=6, title='Scenario', title_fontsize=6, frameon=True, ncol=1, loc='best')
ax.set_xlim(min(model.frange), max(model.frange))
ax.tick_params(axis='x', labelsize=7)
ax.tick_params(axis='y', labelsize=7)

plt.tight_layout(pad=0.3)
plt.savefig("./ps/Fig5_D_Responses.pdf", format="pdf", bbox_inches="tight", pad_inches=0.01)
plt.show()

## Connectivity Matrices

In [ ]:
import glob
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from RRN_functions import RRNModel
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle

# ---- Resolve project root robustly (works even if notebook CWD is not repo root) ----
def _find_project_root(start: Path, marker: str = 'RRN_Simulated.ipynb') -> Path:
    p = start.resolve()
    for cand in [p] + list(p.parents):
        if (cand / marker).exists():
            return cand
    return p

PROJECT_ROOT = _find_project_root(Path.cwd())
WRES_DIR = PROJECT_ROOT / 'out' / 'best_runs_simulated' / 'W_res'
pattern = str(WRES_DIR / 'W_res_RRN_golden_seed*.npy')

# ---- Load freqs (for tick labels) ----
rrn_tmp = RRNModel(n_classes=2, rhythm_configuration='golden', verbose=False)
freqs = np.asarray(rrn_tmp.frange, dtype=float)

# ---- Load W_res files (RRN_golden only; simulated-data runs) ----
files = sorted(glob.glob(pattern))
print('CWD:', Path.cwd())
print('PROJECT_ROOT:', PROJECT_ROOT)
print('W_res glob pattern:', pattern)
print('Found', len(files), 'W_res files')

if not files:
    if WRES_DIR.exists():
        sample = sorted([p.name for p in WRES_DIR.glob('*.npy')])[:10]
        print('WRES_DIR exists but no matches. Example .npy files:', sample)
    raise FileNotFoundError(
        f'No files matched: {pattern}. '
        f'Check arch name and seed naming in {WRES_DIR}.'
    )

Ws = np.stack([np.load(p) for p in files], axis=0)  # (N, K, K)
N, K, _ = Ws.shape

# ---- Summary matrices ----
W_mean = Ws.mean(axis=0)

# Shared symmetric color scale
m = np.max(np.abs(W_mean))
norm = TwoSlopeNorm(vmin=-m, vcenter=0.0, vmax=m)

# Used to analyze weight distributions.
low_pre_idx   = np.where(freqs <= 9)[0]
high_pre_idx  = np.where((freqs >= 20) & (freqs <= 100))[0]
low_post_idx  = np.where(freqs <= 9)[0]
high_post_idx = np.where((freqs >= 20) & (freqs <= 100))[0]

if len(low_pre_idx) == 0 or len(high_pre_idx) == 0:
    raise ValueError("No frequencies found in one of the requested bands.")

# ---- Block means for histogram ----
low_to_high = []
high_to_low = []

for W in Ws:
    low_to_high_block = W[np.ix_(high_post_idx, low_pre_idx)]
    low_to_high.append(np.mean(low_to_high_block))

    high_to_low_block = W[np.ix_(low_post_idx, high_pre_idx)]
    high_to_low.append(np.mean(high_to_low_block))

low_to_high = np.array(low_to_high)
high_to_low = np.array(high_to_low)

bins = np.arange(-0.15, 0.15, 0.02)

# =========================================================
# Figure 1: Connectivity matrix only (2.5 in wide)
# =========================================================
fig1, ax1 = plt.subplots(figsize=(2.5, 2.2), constrained_layout=True)

ax1.imshow(W_mean, aspect="auto", origin="lower", cmap="RdBu_r", norm=norm)

idxs = np.arange(K)
ax1.set_xticks(idxs)
ax1.set_yticks(idxs)
ax1.set_xticklabels([f"{freqs[i]:.1f}" for i in idxs], rotation=90, fontsize=7)
ax1.set_yticklabels([f"{freqs[i]:.1f}" for i in idxs], fontsize=7)

ax1.set_xlabel("Pre frequency (Hz)", fontsize=8)
ax1.set_ylabel("Post frequency (Hz)", fontsize=8)
ax1.tick_params(length=1.5, width=0.5, pad=0.5)

# ---- Box for low-to-high block ----
x0 = low_pre_idx[0] - 0.4
y0 = high_post_idx[0] - 0.5
w  = low_pre_idx[-1] - low_pre_idx[0] + 0.9
h  = high_post_idx.max() - high_post_idx.min() + 1
rect1 = Rectangle((x0, y0), w, h, fill=False, edgecolor='black', linewidth=1)
ax1.add_patch(rect1)

# ---- Box for high-to-low block ----
x0 = high_pre_idx[0] - 0.5
y0 = low_post_idx[0] - 0.4
w  = high_pre_idx.max() - high_pre_idx.min() + 1
h  = low_post_idx.max() - low_post_idx.min() + 1
rect2 = Rectangle((x0, y0), w, h, fill=False, edgecolor='black', linewidth=1)
ax1.add_patch(rect2)

fig1.savefig("./ps/Fig5_E_Connectivity_Matrix.pdf", format="pdf", bbox_inches="tight", pad_inches=0.01)
plt.show()

# =========================================================
# Figure 2: Histogram only (2.5 in wide)
# =========================================================
fig2, ax2 = plt.subplots(figsize=(2.5, 2.0), constrained_layout=True)

ax2.set_axisbelow(True)  # put grid behind histogram bars

ax2.hist(high_to_low, bins=bins, alpha=0.7, label="high→low")
ax2.hist(low_to_high, bins=bins, alpha=0.7, label="low→high")
ax2.set_xlabel("Mean Connectivity", fontsize=8)
ax2.set_ylabel("Count", fontsize=8)
ax2.legend(fontsize=6, frameon=False)
ax2.grid(True, axis="both", alpha=0.5)
ax2.tick_params(axis='both', labelsize=7)

fig2.savefig("./ps/Fig5_F_Block_Means_Hist.pdf", format="pdf", bbox_inches="tight", pad_inches=0.01)
plt.show()

t_stat, p_val = stats.ttest_ind(high_to_low, low_to_high)
print("ttest2 p-value:", p_val)
print("ttest2 stats:", t_stat)